# Your First Hybrid Job

Build a Bell state locally, then package it as a managed Amazon Braket Hybrid Job and walk the create -> queue -> run -> complete lifecycle.

**Objectives:**
- Build and run a parameterized Bell-state circuit on the `LocalSimulator` and read its |00>/|11> correlation
- Write a job entry-point script that builds the circuit and calls `save_job_result`
- Read the `AwsQuantumJob.create(...)` / `job.state()` / `job.result()` lifecycle as code, gated behind `RUN_ON_AWS`
- Explain when a Hybrid Job earns its keep over a standalone task

**Reference:** See [`../GUIDE.md`](../GUIDE.md) for concept explanations and context.

In [ ]:
# Setup: run this cell first.
# Requires: pip install -e '.[dev]' from the project root (see `make setup`).
#
# Importing braket.aws / braket.jobs is free and needs NO AWS credentials.
# Nothing in this notebook calls AWS at import time. Every AWS call below is
# guarded behind `RUN_ON_AWS = False` with a cost note.
import numpy as np
import matplotlib.pyplot as plt

from braket.circuits import Circuit, FreeParameter
from braket.devices import LocalSimulator

# These imports succeed without credentials; we only *call* them under RUN_ON_AWS.
from braket.aws import AwsQuantumJob
from braket.jobs import save_job_result

np.random.seed(7)

# Free, instant, no credentials. All real output in this notebook comes from here.
device = LocalSimulator()
print("LocalSimulator ready:", device)

## 1. Build and run the Bell state locally

Before packaging anything for AWS, prove the circuit on your laptop. The Bell state
$|\Phi^+\rangle = \tfrac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$ is the canonical two-qubit entangled state: a Hadamard on qubit 0 puts it in superposition, then a CNOT copies that into qubit 1. Measuring it yields only `00` or `11` -- never `01` or `10` -- because the two qubits are perfectly correlated.

In [ ]:
bell = Circuit().h(0).cnot(0, 1)
print(bell)

result = device.run(bell, shots=1000).result()
counts = dict(result.measurement_counts)
print("\ncounts:", counts)

# The defining property: only the correlated outcomes appear.
assert set(counts).issubset({"00", "11"}), counts
print("Only |00> and |11> observed -- the qubits are entangled.")

In [ ]:
# Visualize the correlation.
fig, ax = plt.subplots(figsize=(4.5, 3))
labels = ["00", "01", "10", "11"]
values = [counts.get(b, 0) for b in labels]
ax.bar(labels, values, color=["#33bb66", "#cccccc", "#cccccc", "#33bb66"])
ax.set_xlabel("measured bitstring")
ax.set_ylabel("counts (1000 shots)")
ax.set_title("Bell state: only |00> and |11>")
plt.tight_layout()
plt.show()

## 2. Parameterize it -- the inner loop of every hybrid job

A real hybrid job does not submit a fixed circuit once; it submits the *same circuit
structure* every iteration with a fresh angle chosen by a classical optimizer. Declare
the angle as a `FreeParameter` and pass values through `inputs={...}` at run time. This
is the seed of *parametric compilation*: one structure, many parameter values.

Here `Rx(theta)` on qubit 0 followed by a CNOT sweeps continuously from the product
state `00` (theta=0), through a balanced entangled state (theta=pi/2), to the flipped
product state `11` (theta=pi).

In [ ]:
theta = FreeParameter("theta")
param_bell = Circuit().rx(0, theta).cnot(0, 1)
print(param_bell)

for t in [0.0, np.pi / 2, np.pi]:
    r = device.run(param_bell, shots=2000, inputs={"theta": float(t)}).result()
    print(f"theta={t:5.3f}  ->  {dict(r.measurement_counts)}")

## 3. A local variational loop -- what the job will actually run

The classical half of a hybrid algorithm reads a measurement, computes a cost, and picks
the next angle. We run that loop here locally so you see real convergence with no AWS.

The cost is $\langle Z_0 \rangle$ -- the expectation of $Z$ on qubit 0, estimated from
shot counts as $(n_0 - n_1)/N$. Minimizing it drives qubit 0 toward $|1\rangle$, i.e.
$\theta \to \pi$. We use a finite-difference gradient and plain gradient descent. This
is intentionally tiny (1 qubit, 12 steps) so it runs in a second -- but it is structurally
the same loop a VQE or QAOA job runs for hundreds of iterations.

In [ ]:
one_qubit = Circuit().rx(0, FreeParameter("theta"))

def expval_z0(angle, shots=4000):
    """Estimate <Z0> from shot counts: (#|0> - #|1>) / N."""
    c = dict(device.run(one_qubit, shots=shots, inputs={"theta": float(angle)}).result().measurement_counts)
    n0, n1 = c.get("0", 0), c.get("1", 0)
    return (n0 - n1) / (n0 + n1)

angle = 0.3          # initial guess
lr = 0.6             # learning rate
eps = 0.1            # finite-difference step
cost_history = []

for step in range(12):
    grad = (expval_z0(angle + eps) - expval_z0(angle - eps)) / (2 * eps)
    cost = expval_z0(angle)
    cost_history.append(cost)
    angle = angle - lr * grad

print("final theta:", round(angle, 3), " target pi =", round(float(np.pi), 3))
print("cost went from", round(cost_history[0], 3), "to", round(cost_history[-1], 3))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(range(len(cost_history)), cost_history, "o-", color="#3366cc")
ax.axhline(-1.0, ls="--", color="#999999", label="ground state <Z0> = -1")
ax.set_xlabel("iteration")
ax.set_ylabel("cost  <Z0>")
ax.set_title("Local variational loop converges")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Package the algorithm: the entry-point script

A Hybrid Job runs an **entry-point script** inside a managed container on AWS. The script
is ordinary Python: it builds the circuit, runs it (on the job's device), and calls
`save_job_result(...)` to persist outputs to S3. Inside a real job the device comes from
the `AMZN_BRAKET_DEVICE_ARN` environment variable that Braket injects; locally we fall
back to `LocalSimulator` so the *same script* is runnable on your laptop.

We define the entry point as a **string** and write it to a file. Defining and compiling a
string touches no AWS -- we even run it locally below to prove it works before any job exists.

In [ ]:
ENTRY_POINT = """
import os

from braket.circuits import Circuit
from braket.devices import LocalSimulator
from braket.jobs import save_job_result


def get_device():
    # Inside a Hybrid Job, Braket sets AMZN_BRAKET_DEVICE_ARN. Run the same
    # script locally by falling back to the LocalSimulator when it is absent.
    arn = os.environ.get("AMZN_BRAKET_DEVICE_ARN")
    if arn and arn.startswith("arn:"):
        from braket.aws import AwsDevice
        return AwsDevice(arn)
    return LocalSimulator()


def main():
    shots = int(os.environ.get("SHOTS", "1000"))
    device = get_device()

    bell = Circuit().h(0).cnot(0, 1)
    counts = dict(device.run(bell, shots=shots).result().measurement_counts)

    p_correlated = (counts.get("00", 0) + counts.get("11", 0)) / shots
    save_job_result({"counts": counts, "p_correlated": p_correlated})
    print("counts:", counts, "p_correlated:", p_correlated)


if __name__ == "__main__":
    main()
"""

with open("first_job_entry.py", "w") as f:
    f.write(ENTRY_POINT)

print("Wrote first_job_entry.py (", len(ENTRY_POINT), "chars )")

In [ ]:
# Sanity-check the entry point WITHOUT AWS: run it as a subprocess against the
# LocalSimulator (AMZN_BRAKET_DEVICE_ARN unset -> falls back to LocalSimulator).
# save_job_result is a no-op outside a real job, so this exercises the real code path
# end-to-end with no credentials.
import os
import subprocess
import sys

proc = subprocess.run(
    [sys.executable, "first_job_entry.py"],
    capture_output=True, text=True,
    env={**os.environ, "SHOTS": "500"},
)
print("exit code:", proc.returncode)
print(proc.stdout.strip())
if proc.returncode != 0:
    print("STDERR:", proc.stderr[-500:])

## 5. Submit it as a Hybrid Job (gated -- needs AWS)

This is the only part that touches AWS. `AwsQuantumJob.create(...)` packages the entry
point, uploads it, provisions a classical instance, and runs your script with **priority
access** to the device. You then poll `job.state()` through the lifecycle and pull
`job.result()` from S3 when it reaches `COMPLETED`.

The lifecycle is fixed: **create -> QUEUED -> RUNNING -> COMPLETED** (or `FAILED` /
`CANCELLED`). Everything below is guarded behind `RUN_ON_AWS = False` so this cell is inert
by default.

In [ ]:
RUN_ON_AWS = False  # Flip to True ONLY with AWS credentials configured.

# COST NOTE: A Hybrid Job bills the classical instance per hour (ml.m5.large ~ $0.10-$0.30/hr)
# PLUS the quantum tasks. On the local device below the quantum side is free and the run is
# seconds, so cost is dominated by the brief instance time. Switching `device` to a real QPU
# ARN adds per-shot + per-task charges. Always prototype locally first (see cells 1-4).

if RUN_ON_AWS:
    import time

    job = AwsQuantumJob.create(
        device="local:braket/braket.local.qubit",  # free local device inside the managed instance
        source_module="first_job_entry.py",
        entry_point="first_job_entry:main",
        job_name="first-hybrid-bell",
        hyperparameters={},  # the script reads SHOTS from the environment
        wait_until_complete=False,
    )
    print("created:", job.arn)

    # Poll the lifecycle: QUEUED -> RUNNING -> COMPLETED.
    terminal = {"COMPLETED", "FAILED", "CANCELLED"}
    state = job.state()
    while state not in terminal:
        print("state:", state)
        time.sleep(15)
        state = job.state()
    print("final state:", state)

    if state == "COMPLETED":
        result = job.result()  # downloads from S3
        print("result:", result)
else:
    print("RUN_ON_AWS is False -- no AWS calls made.")
    print("Local results above already prove the circuit and the entry point work.")

## 6. When does a job beat a standalone task?

A standalone `device.run(circuit)` task is fire-and-forget: it joins the back of the
device's general queue. For one circuit that is fine. The break-even is **iteration**. A
variational loop where each step depends on the last pays the queue wait *once per step* as
a standalone task, with the classical optimizer idle in between. A Hybrid Job pays the
queue wait essentially once: its tasks carry a job token granting priority, so iterations
run back-to-back.

Rough rule of thumb: `standalone_wall ~ iterations * (queue_wait + run)` versus
`job_wall ~ startup + iterations * run`. The job's fixed startup is amortized once
iterations and queue wait are large.

In [ ]:
def standalone_wall(iterations, queue_wait, run_sec):
    return iterations * (queue_wait + run_sec)

def job_wall(iterations, run_sec, startup=180.0):
    # priority access -> queue paid ~once, folded into the fixed startup
    return startup + iterations * run_sec

iters = np.arange(1, 201)
queue_wait, run_sec = 45.0, 6.0
standalone = np.array([standalone_wall(n, queue_wait, run_sec) / 60 for n in iters])
job = np.array([job_wall(n, run_sec) / 60 for n in iters])

fig, ax = plt.subplots(figsize=(5.5, 3.2))
ax.plot(iters, standalone, label="standalone tasks", color="#cc3333")
ax.plot(iters, job, label="hybrid job", color="#3366cc")
ax.set_xlabel("iterations")
ax.set_ylabel("wall-clock (minutes)")
ax.set_title("Job wins once iteration count is high")
ax.legend()
plt.tight_layout()
plt.show()

# Break-even iteration count.
crossover = int(iters[np.argmax(standalone > job)])
print("standalone overtakes the job at ~", crossover, "iterations (queue_wait=45s, run=6s)")

## Exercises

Work these locally first; only flip `RUN_ON_AWS` with credentials and a cost check.

In [ ]:
# Exercise 1: Sweep theta from 0 to pi in 9 steps on the parameterized Bell circuit and
# plot p(correlated) = p(00) + p(11) versus theta. Where is the entanglement strongest?
# TODO: reuse `param_bell` and inputs={"theta": ...}.

# Exercise 2: Change the local variational loop's cost to <Z0 Z1> (the Bell correlation).
# Estimate it as (n00 + n11 - n01 - n10) / N from a two-qubit circuit and re-run the descent.
# TODO: build Circuit().rx(0, theta).cnot(0, 1), define expval_zz, reuse the loop in cell 3.

# Exercise 3: Extend ENTRY_POINT to read a hyperparameter n_layers from the environment,
# build that many H+CNOT layers, and save the layer count alongside the counts.
# Re-run the local subprocess check to confirm it still works with no AWS.
# TODO: edit ENTRY_POINT, write the file, re-run the subprocess cell.

# Exercise 4 (needs AWS, cost-gated): With RUN_ON_AWS=True, submit the job, then while it
# runs print job.state() every 15s and time how long QUEUED -> RUNNING -> COMPLETED takes.
# TODO: instrument the polling loop in cell 5 with time.time() stamps.

## Summary

- A Bell state (`H` then `CNOT`) produces only the correlated outcomes `|00>` and `|11>`; the parameterized `Rx(theta)+CNOT` version is the inner loop a hybrid job iterates.
- Everything that produced output here ran on the `LocalSimulator` with no AWS credentials -- the circuit, the `FreeParameter` sweep, the variational descent, and the entry point (exercised as a subprocess).
- A Hybrid Job is just your entry-point script (which calls `save_job_result`) handed to `AwsQuantumJob.create(...)`; you then poll `job.state()` through create -> QUEUED -> RUNNING -> COMPLETED and pull `job.result()` from S3. All AWS calls stay behind `RUN_ON_AWS` with a cost note.
- A job earns its keep on **iteration**: many quantum-classical steps with real queue wait, where priority access lets iterations run back-to-back. A single circuit belongs in a standalone task.

**Next:** [`02-parametric-compilation.ipynb`](02-parametric-compilation.ipynb) -- compile the parameterized circuit once and reuse it across parameter updates to cut per-iteration time.